

## Learning Objectives

By the end of this notebook, you will be able to:

1. Create and manipulate NumPy arrays
2. Understand the performance benefits of vectorization
3. Apply vectorized operations instead of loops
4. Use NumPy's built-in functions for common operations

## Introduction

This activity uses NumPy, the standard library for numerical arrays in Python. Recall from class that NumPy's central data structure is `ndarray`, an N-dimensional array containing values of the same type (`dtype` in NumPy). These N-dimensional arrays are also referred to as ["tensors"](https://en.wikipedia.org/wiki/Tensor_(machine_learning)). A 1-D array is often called a "vector" and is analogous to R's atomic vector (a single element is referred to as a "scalar"). A 2-D array is often called a "matrix" and is analogous to R's matrix. NumPy arrays can have more than 2 dimensions.

::: {.callout-tip collapse="true"}
## Coming from R? Some tips.

In many respects Python and R are very similar, especially when using libraries like NumPy and Pandas, which share inspiration with their R counterparts. Both are dynamically typed languages. Both offer similar semantic constructs like functions, loops, conditionals. Two immediate differences you will notice: Python uses indentation, not braces, to delineate blocks and Python is 0-indexes, i.e., sequences start at index 0, instead of at index 1 as in R.

R was specifically designed for data analysis and so types like matrices, data frames, etc. are built into the language. In contrast Python is a more "general purpose" language and so those types are provided via external libraries. As a result there can sometime be interface "friction" as those libraries have to work within the features provided by Python.

In these early notebooks we will include additional comments/notes to support your transition from R to Python. Keep an eye out!
:::

As described in class, when using NumPy we want to start thinking in "vector" or "array" operations, i.e., performing computations across multiple values instead of using loops. That way we make best use of NumPy's high-performance native implementations (e.g., implemented in C, not Python). Idiomatic NumPy code, like idiomatic R code, often has not explicit loops.

NumPy et al. are massive libraries. No one can or should know the functions that are available. Recall we are trying to cultivate a sense of how to approach problems with these tools, and what kind of functions should be available. We can then search the documentation, use GenAI tools, Python's built in `help` function, etc. to find and use the relevant functions.

Get started by importing the relevant libraries:

In [ ]:
# This is the typical import style/prefix. It makes all the NumPy types and functions
# available with a `np` prefix.
import numpy as np

print("NumPy version:", np.__version__)

## Array creation and manipulation

A [random walk](https://en.wikipedia.org/wiki/Random_walk) is a path composed of a succession of random steps, e.g. that path of molecule as it travels in a liquid of gas. Random walks have many application in a wide variety of fields from the physical sciences to finance! Here we will consider a simple 1-D walk [@mckinneyPythonDataAnalysis2022].

In [ ]:
# Create a 1-D random walk of +1 or -1 steps
steps = 100
# Randomly choose -1 or 1 `steps` times, producing 1-D integer array of length steps
delta = np.random.choice([-1, 1], steps)
# Compute the resulting walk as the cumulative sum of those steps, i.e., index i is the sum of 0..i-1 steps
walk = np.cumsum(delta)
print(walk) # Print the walk

We will learn more about plotting shortly. For now, just treat the plot as a given.

In [ ]:
from matplotlib import pyplot as plt
plt.plot(range(0, len(walk)), walk)

The above is just a single a walk. We typically perform this simulation many times, e.g., 5000, to get a distribution of outcomes. Here is where the vectorized approach can be particularly helpful (for both conciseness and performance). We can think of 5000 random walks of length 100 as working with a 2-D array with 5000 rows and 100 columns where each row is different walk and each column is a time step.

### What you should do

1. The code below creates a 2-D array of size 5000x100. Complete function below to by defining a `walks` variable that computes the position of each walk at each time point. We can use the same [`cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html) function, but note we need to specify which dimension or `axis` we want to perform the sum across. If we don't we will get the cumulative sum of the entire entire array "flattened" into a 1-D vector. The axes are numbered according to the shape, i.e., index 0 of the shape is axis 0, index 1 is axis 1, ... If the shape is `(5000, 100)`, summing across axis 1 sums across the 100 dimension (i.e., "across" the columns or "along" the rows).

In [ ]:
def sim_walks(num_steps, num_walks): # <1>
    """Simulate num_walks 1-D random walks of num_steps each"""
    delta = np.random.choice([-1, 1], (num_walks, num_steps)) # Specify size as 2-tuple (# rows, # columns)
    walks = None
    # 
    return walks # <2>

walks = sim_walks(100, 5000)
print(walks.shape == (5000, 100)) # Verify the expected shape

   1. `def` is used to define a function, here named `sim_walks` with two positional arguments `num_steps` and `num_walks`. The body of function is indented (typically 4 spaces, but any indentation is allowed as long as it is consistent).
   2. Python requires an explicit `return` statement (unlike R which implicitly returns the value of the last expression in a function), which terminates execution and "returns" a value to the caller. Functions without a `return` statement are permitted. In that case, the function implicitly returns a special value `None`.
2. Replace the `pass` in the function below to return the distance of the most distant ending point (from the origin) from across all the walks. Think about what portions of the `walks` array might be relevant answering that question. For context, your answer can be a single line.

In [ ]:
def farthest_end(walks):
    """Return distance of most distant ending point in (walks, steps) walks array"""
    pass
    # 

farthest_end(walks)

3. Replace the `pass` in the function below to return the fraction of steps in positive territory (> 0).

In [ ]:
def fraction_positive(walks):
    """Return fraction of steps in walks in positive territory (> 0)"""
    pass
    # 

fraction_positive(walks)

4. Select all the minimum NumPy features needed to implement `farthest_end` and `fraction_positive`. Multiple values can be selected by holding the ctrl⌃ (or command⌘) key.

In [ ]:
import ipywidgets as widgets
techniques = widgets.SelectMultiple(
    options=[('Vectorized operations', 1), ('Numeric indexing', 2), ('Logical indexing', 3), ('Broadcasting', 4), ('Reduction over specific axis', 5)],
    description='',
    disabled=False
)
display(techniques)

## Translate "plain" Python to Numpy

Mean squared error (MSE), is, as its name suggests, the measure of the average squared difference between the predicted and true values. It is commonly used to evaluate model predictions, including as the loss function to optimize (minimize) when training a prediction model. MSE between true values $y$ and predictions $\hat{y}$ is defined as:

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

A "plain" Python implementation for MSE is below. 

In [ ]:
def mse_plain(y_true, y_pred):
    """Compute MSE using 'plain' Python for equal length sequences y_true and y_pred"""
    n = len(y_true)
    total = 0.0
    for i in range(n): # <1>
        diff = y_true[i] - y_pred[i]
        # total += ... equivalent to total = total + ...
        total += diff * diff 
    return total / n

1. A python `for` loop defines a loop variable (here `i`) that is assigned values from from a loop sequence. Here `range(n)` generates a sequence from 0 to `n`, counting by 1 (e.g., 0, 1, 2, ..., `n-1`).

The equivalent R code is shown below. Notice the near one-to-one semantic concordance...
```r
mse_plain <- function(y_true, y_pred) {
    n <- length(y_true)
    total <- 0.0
    for (i in seq_len(n)) {
        diff <- y_true[i] - y_pred[i]
        total <- total + diff * diff
    }
    total / n
}
```

::: {.callout-tip collapse="true"}
## That is not how I would implement it in R

Us neither! Stay tuned for how NumPy allows for a "vectorized" approach similar to idiomatic R.
:::

### What You Should Do
1. Convince yourself that this code implements the equation above.
2. Implement a "vectorized" version of MSE using NumPy as the function `mse_numpy` (replacing the call to `mse_plain` with your code). You can assume the inputs are NumPy arrays. *Your function should not have any loops*. For context, your implementation can be a single line!

In [ ]:
def mse_numpy(y_true, y_pred):
    """Compute MSE between y_true and y_pred NumPy arrays without loops"""
    return mse_plain(y_true, y_pred)
    # your code here

3. Use the following code to verify your implementation matches the original. Note that we don't perform a strict equality test here. Floating point arithmetic has finite precision and so the two implementations might produce slightly different results (why we generally don't use strict equality testing for floating point values).

In [ ]:
n = 1000000
y_true = np.random.normal(size=n)
y_pred = y_true + np.random.normal(scale=0.1, size=n)

plain_mse = mse_plain(y_true, y_pred)
numpy_mse = mse_numpy(y_true, y_pred)

# Verify arrays match within some tolerance
np.allclose(plain_mse, numpy_mse, atol=1e-12)

4. Which best describes the relative execution time of the two implementations? Try answering before running the benchmark code below, i.e., make a prediction then update your answer based on the actual results.

In [ ]:
import ipywidgets as widgets
perf_alt = widgets.RadioButtons(
    options=[('They take the same time', 1), ('The plain version is faster', 2), ('The NumPy version is faster', 3)],
    description=''
)
display(perf_alt)

In [ ]:
%timeit mse_plain(y_true, y_pred)

In [ ]:
%timeit mse_numpy(y_true, y_pred)

5. Which best describes the relative big-O time complexity of the two implementations? [Big-O](https://en.wikipedia.org/wiki/Big_O_notation) is used to describe how the run time (or other resource consumed by an algorithm) grows as the size of the input, typically noted as $n$, grows arbitrarily large. Big-O provides a helpful analytical tool for evaluating the performance of algorithms without needing to implement them (as we did above). We can determine the big-O time complexity by counting the number of operations required as function of the input size, $f(n)$, the dropping all coefficients and all but the fastest growing term. For example, $f(n)=2n^2+3$ implies $\mathcal{O}=n^2$.

In [ ]:
import ipywidgets as widgets
complex_alt = widgets.RadioButtons(
    options=[('They have the same time complexity', 1), ('The plain version has lower time complexity', 2), ('The NumPy version has lower time complexity', 3)],
    description=''
)
display(complex_alt)

6. Briefly explain why your answers to the two questions above are consistent. What does your answer imply for thinking about the performance of our code.